In [52]:
#load data zwn
import pandas as pd
raw_zwn = pd.read_csv("../data/raw/zwn_meldungen.csv")

#Clean data
#select useful columns
processed_zwn = raw_zwn[["service_name","requested_datetime","e","n","status","updated_datetime"]]


#define new columns names
new_names = {
"service_name":"category",
"e":"East",
"n":"North",
"requested_datetime":"report_time",
"updated_datetime":"resolved_time",
    }
processed_zwn= processed_zwn.rename(columns=new_names)

# convert datatype of "report_time" to datetime64
processed_zwn["report_time"] = pd.to_datetime(processed_zwn["report_time"], format ="%Y-%m-%dT%H:%M:%S")
processed_zwn["resolved_time"] = pd.to_datetime(processed_zwn["resolved_time"], format ="%Y-%m-%dT%H:%M:%S")

# create geometry category and geodataframe
import geopandas as gpd
from shapely.geometry import Point
processed_zwn = gpd.GeoDataFrame(processed_zwn,geometry=gpd.points_from_xy(processed_zwn["East"], processed_zwn["North"])
)
# select useful columns
processed_zwn= processed_zwn[["category","report_time","geometry","status","resolved_time"]]


# check missing values
#missing_count = processed_zwn[["category","report_time","geometry"]].isna().sum()
#print(f"The table shows the missing values in the dataframe {missing_count}")

#define missing CRS (CH1903+ / LV95)
processed_zwn = processed_zwn.set_crs(epsg=2056)

# load data quartiere
raw_quartiere = pd.read_csv("../data/raw/quartiere_zürich.csv")
processed_quartiere = raw_quartiere[["qname","geometry"]]
processed_quartiere.head(3)

#define new columns names
new_names1 = {
"qname":"Quartier",
    "geometry": "Geometry"
}
processed_quartiere= processed_quartiere.rename(columns=new_names1)

# check missing values
#missing_count = processed_quartiere[["Quartier","Geometry"]].isna().sum()
#print(f"The table shows the missing values in the dataframe {missing_count}")

#transform geometry datatype
#wkt.loads transform string datatype to geometry data type
from shapely import wkt
processed_quartiere["Geometry"] = processed_quartiere["Geometry"].apply(wkt.loads)

# create geodataframe to interprate "Geometry" as a geometry column
processed_quartiere = gpd.GeoDataFrame(
    processed_quartiere,
    geometry="Geometry")

#define missing CRS (CH1903+ / LV95)
processed_quartiere = processed_quartiere.set_crs(epsg=2056)

#download processed csv-files
#processed_zwn.to_file("../data/processed/processed_zwn.gpkg", driver="GPKG")
#processed_quartiere.to_file("../data/processed/processed_quartiere.gpkg", driver="GPKG")

# Question 4: "Wie unterscheidet sich die Bearbeitungszeit der Quartiere pro Kategorie ? "

#perform spatial join
zwn_with_quartiere = gpd.sjoin(processed_zwn,processed_quartiere, how="left", predicate ="within")

#filter data
#filter only reports which are already fixed
zwn_with_quartiere_filtred= zwn_with_quartiere[zwn_with_quartiere["status"] =="fixed - council"]
zwn_with_quartiere_category= zwn_with_quartiere_filtred[zwn_with_quartiere_filtred["category"]=="Strasse/Trottoir/Platz"]

#calculate processing time
zwn_with_quartiere_category["processing_time"] = zwn_with_quartiere_category["resolved_time"]-zwn_with_quartiere_category["report_time"]


#calculate mean processing time per Quartier
mean = (zwn_with_quartiere_category.groupby(["Quartier"])["processing_time"].mean().reset_index())



    



,Quartier,processing_time
0,Affoltern,3 days 12:01:19.662650
1,Albisrieden,3 days 03:06:31.669144
2,Alt-Wiedikon,11 days 00:23:54.934579
3,Altstetten,2 days 20:22:53.093555
4,City,3 days 16:06:22.254355
5,Enge,2 days 21:02:19.794117
6,Escher Wyss,3 days 20:41:50.842443
7,Fluntern,2 days 17:12:34.008547
8,Friesenberg,3 days 10:28:25.389473
9,Gewerbeschule,2 days 23:57:03.098540
